# MinHash and LSH

This notebook builds lexical candidate pairs from question text. We search all questions, including those in the test split, but use `dev` for every parameter choice and metric.

For q=1, each question is a set of words, matching the Jaccard definition in P1. The q=2 run is a comparison: it uses sets of adjacent word pairs.

In [1]:
import gc
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")
from src.evaluate import evaluate, recall_by_jaccard_bin
from src.minhash import minhash_signatures, lsh_candidates

DATA_DIR = Path("../data/processed")
questions = pd.read_parquet(DATA_DIR / "questions.parquet")
pairs = pd.read_parquet(DATA_DIR / "pairs.parquet")

print(f"Questions: {len(questions):,}; labelled pairs: {len(pairs):,}")
print("Pair labels by split:\n", pairs.groupby(["split", "is_duplicate"]).size())

Questions: 537,174; labelled pairs: 403,772
Pair labels by split:
 split  is_duplicate
dev    0               204500
       1               118931
test   0                50499
       1                29842
dtype: int64


## Method

`shingles(text, q)` lowercases text, extracts `\w+` tokens, and keeps each distinct q-word shingle once. `minhash_signatures` hashes those shingles with 96 affine hash functions generated from a fixed seed. It works in batches to keep memory use bounded.

The signature is split into `bands` chunks of `rows` values each (`k = bands × rows`). Two questions become candidates if any band matches. Their `score` is the fraction of matching values in the full signatures, an estimate of Jaccard similarity.

For a pair with similarity `s`, the chance of becoming a candidate is `1 - (1 - s**rows)**bands`. The full-data comparison below checks how that trade-off works on this dataset; the setting with the highest dev F1 is selected.

In [2]:
# Quick check on the 10% sample.

sample_questions = pd.read_parquet(DATA_DIR / "questions_sample10.parquet")
sample_pairs = pd.read_parquet(DATA_DIR / "pairs_sample10.parquet")

sample_signatures = minhash_signatures(sample_questions, q=1, num_hashes=96, seed=42)

sample_rows = []
for bands, rows in [(6, 16), (8, 12), (12, 8), (16, 6)]:
    candidates = lsh_candidates(
        sample_signatures, 
        qids=sample_questions["qid"].to_numpy(), 
        bands=bands,
        rows=rows
    )
    result = evaluate(candidates, sample_pairs, sample_questions, split="dev")
    sample_rows.append({"bands": bands, "rows": rows, **result})
    del candidates
    gc.collect()

sample_study = pd.DataFrame(sample_rows)
sample_study[["bands", "rows", "n_candidates", "recall", "precision", "f1", "recall_implied"]]

,bands,rows,n_candidates,recall,precision,f1,recall_implied
0,6,16,3861,0.068556,0.570278,0.122398,0.052474
1,8,12,10711,0.117751,0.542799,0.193520,0.090535
2,12,8,51650,0.233741,0.546390,0.327416,0.182066
3,16,6,278597,0.350788,0.539451,0.425128,0.277317


## Full-data sweep: q=1

Every question gets a signature, including questions in the test split. We do not use test labels for tuning. The full-data sweep compares some factorizations of the fixed signature length `k=96`. Only full-data results are reported as project metrics.

In [3]:
NUM_HASHES = 96
SEED = 42
Q = 1

started = time.perf_counter()

signatures_q1 = minhash_signatures(questions, q=Q, num_hashes=NUM_HASHES, seed=SEED)

print(
    f"q={Q} signatures: {signatures_q1.shape}; "
    f"questions without shingles: {(signatures_q1[:, 0] == 2**32 - 1).sum():,}; "
    f"time: {time.perf_counter() - started:.1f}s"
)

q=1 signatures: (537174, 96); questions without shingles: 6; time: 16.9s


In [4]:
# Evaluate each setting on dev and keep only the current best candidate set in memory.

parameter_grid = [(6, 16), (8, 12), (12, 8), (16, 6)]
full_rows = []
best_candidates = None
best_f1 = -1.0
best_config = None

for bands, rows in parameter_grid:
    started = time.perf_counter()

    candidates = lsh_candidates(
        signatures_q1,
        qids=questions["qid"].to_numpy(),
        bands=bands,
        rows=rows,
    )
    metrics = evaluate(candidates, pairs, questions, split="dev")
    full_rows.append(
        {
            "q": Q,
            "num_hashes": NUM_HASHES,
            "bands": bands,
            "rows": rows,
            **metrics,
            "seconds": time.perf_counter() - started,
        }
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_config = (bands, rows)
        best_candidates = candidates
    else:
        del candidates
    gc.collect()

full_study_q1 = pd.DataFrame(full_rows).sort_values("f1", ascending=False).reset_index(drop=True)

print(full_study_q1[[
    "q", "num_hashes", "bands", "rows", "n_candidates", "fraction_of_all_pairs",
    "labelled_candidates", "recall", "precision", "f1", "recall_implied", "seconds"
]].to_string(index=False))

 q  num_hashes  bands  rows  n_candidates  fraction_of_all_pairs  labelled_candidates   recall  precision       f1  recall_implied    seconds
 1          96     16     6      26906047               0.000186                69784 0.306152   0.521767 0.385883        0.208893 191.239180
 1          96     12     8       4305202               0.000030                45063 0.192507   0.508066 0.279218        0.128987  31.096710
 1          96      8    12        735804               0.000005                22824 0.092432   0.481642 0.155099        0.061135   9.637564
 1          96      6    16        196541               0.000001                12704 0.052207   0.488744 0.094337        0.034461   6.524148


In [5]:
best_bins_q1 = recall_by_jaccard_bin(best_candidates, pairs, questions, split="dev")

print(f"Selected q=1 setting on dev: bands={best_config[0]}, rows={best_config[1]}")
best_bins_q1

Selected q=1 setting on dev: bands=16, rows=6


,n_pairs,recall
bin,,
"(-0.001, 0.2]",9197,0.000326
"(0.2, 0.4]",42466,0.020087
"(0.4, 0.6]",36385,0.228446
"(0.6, 0.8]",21819,0.833494
"(0.8, 1.0]",9064,0.999228


## Compare q=2 and save candidates

After selecting the q=1 banding on `dev`, we run q=2 with the same `k`, bands, and rows. This is a focused comparison, not a separate q=2 parameter sweep. The better of the two runs is saved for P4 in the shared candidate format.

In [6]:
started = time.perf_counter()

signatures_q2 = minhash_signatures(questions, q=2, num_hashes=NUM_HASHES, seed=SEED)
q2_candidates = lsh_candidates(
    signatures_q2,
    qids=questions["qid"].to_numpy(),
    bands=best_config[0],
    rows=best_config[1],
)
q2_metrics = evaluate(q2_candidates, pairs, questions, split="dev")

q2_row = {
    "q": 2,
    "num_hashes": NUM_HASHES,
    "bands": best_config[0],
    "rows": best_config[1],
    **q2_metrics,
    "seconds": time.perf_counter() - started,
}

comparison = pd.concat(
    [full_study_q1.iloc[[0]], pd.DataFrame([q2_row])], ignore_index=True
)
print(comparison[["q", "num_hashes", "bands", "rows", "n_candidates", "fraction_of_all_pairs",
            "recall", "precision", "f1", "recall_implied", "seconds"]].to_string(index=False))

 q  num_hashes  bands  rows  n_candidates  fraction_of_all_pairs   recall  precision       f1  recall_implied    seconds
 1          96     16     6      26906047               0.000186 0.306152   0.521767 0.385883        0.208893 191.239180
 2          96     16     6       2773652               0.000019 0.106314   0.432392 0.170665        0.070561  48.936112


In [7]:
if q2_row["f1"] > best_f1:
    best_candidates = q2_candidates
    best_config_q = 2
else:
    best_config_q = 1
    del q2_candidates
    gc.collect()

output_path = DATA_DIR / "lsh_candidates.parquet"
best_candidates.to_parquet(output_path, index=False, compression="zstd")

print(f"Final choice: q={best_config_q}, k={NUM_HASHES}, bands={best_config[0]}, rows={best_config[1]}")
print(f"Saved candidates: {len(best_candidates):,} to {output_path}")
print("Columns:", best_candidates.columns.tolist())

Final choice: q=1, k=96, bands=16, rows=6
Saved candidates: 26,906,047 to ../data/processed/lsh_candidates.parquet
Columns: ['qid_a', 'qid_b', 'score', 'source']


## What the results tell us

The selected q=1 run finds most duplicate pairs whose word sets are already very similar, and few of the pairs with low Jaccard similarity. That is the gap the embedding and graph stages are meant to explore.

The saved Parquet file contains candidates from searches over the full question set. All reported metrics above use `dev`; the test labels remain reserved for the final report.

In [8]:
# Final evaluation on the test, tuning was on dev
final_candidates = pd.read_parquet(DATA_DIR / "lsh_candidates.parquet")

test_metrics = evaluate(final_candidates, pairs, questions, split="test")
print(pd.Series(test_metrics))

test_bins = recall_by_jaccard_bin(final_candidates, pairs, questions, split="test")
test_bins

n_candidates             2.690605e+07
fraction_of_all_pairs    1.864879e-04
labelled_candidates      1.708900e+04
recall                   2.862744e-01
precision                4.999122e-01
f1                       3.640664e-01
recall_implied           1.892428e-01
dtype: float64


,n_pairs,recall
bin,,
"(-0.001, 0.2]",2726,0.000000
"(0.2, 0.4]",11274,0.018804
"(0.4, 0.6]",8709,0.234355
"(0.6, 0.8]",5137,0.836091
"(0.8, 1.0]",1996,0.999499
